# 02 - Data Cleaning

Clean and prepare the World Bank indicator data for analysis.

**Objectives:**
- Standardize column names
- Create analysis-ready data structures
- Handle missing values
- Export cleaned datasets

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from load_data import load_all_indicators, PROCESSED_DIR
from analysis import calculate_financing_sources, generate_summary_stats

# Create processed directory if it doesn't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries loaded. Processed directory ready.")

## 1. Load Raw Data

In [ ]:
# Load all indicator files
indicators = load_all_indicators()
print(f"Loaded {len(indicators)} indicator files.")

## 2. Create Clean Summary Table

In [ ]:
# Combine all finance-related indicators into a single clean table
finance_df = indicators.get('Viet Nam Finance')

# Clean column names
finance_clean = finance_df.copy()
finance_clean.columns = [
    'indicator_name',
    'indicator_code',
    'vietnam',
    'east_asia_pacific',
    'all_economies'
]

# Add a column showing if Vietnam is above/below global average
finance_clean['vs_global'] = finance_clean.apply(
    lambda row: 'Above' if row['vietnam'] > row['all_economies'] else 'Below',
    axis=1
)

print("Finance Indicators - Cleaned")
display(finance_clean)

## 3. Create Financing Sources Table

In [ ]:
# Get investment financing sources
investment_df = indicators.get('Viet Nam Investment Financing')
sources_df = calculate_financing_sources(investment_df)

# Clean column names
sources_df.columns = ['source', 'vietnam', 'east_asia_pacific', 'all_economies']

print("Investment Financing Sources - Cleaned")
display(sources_df)

## 4. Create Key Metrics Dictionary

In [ ]:
# Extract key metrics as a dictionary for easy access
key_metrics = {
    'credit_constraints': {
        'fully_constrained': 8.2,
        'partially_constrained': 18.0,
        'total': 26.2,
        'vietnam': 26.2,
        'east_asia': 20.9,
        'global': 31.0
    },
    'banking_access': {
        'with_account': 54.8,
        'with_loan': 40.5,
        'vietnam_account': 54.8,
        'east_asia_account': 85.5,
        'global_account': 89.9
    },
    'collateral': {
        'require_collateral': 93.5,
        'vietnam': 93.5,
        'east_asia': 64.4,
        'global': 67.1
    },
    'rejection_rate': {
        'rejected': 1.9,
        'vietnam': 1.9,
        'east_asia': 3.0,
        'global': 6.5
    }
}

print("Key Metrics Dictionary Created:")
for category, values in key_metrics.items():
    print(f"\n{category.upper()}:")
    for key, val in values.items():
        print(f"  {key}: {val}")

## 5. Export Cleaned Data

In [ ]:
# Export cleaned datasets
finance_clean.to_csv(PROCESSED_DIR / 'finance_indicators_clean.csv', index=False)
sources_df.to_csv(PROCESSED_DIR / 'financing_sources_clean.csv', index=False)

# Also save as Excel
with pd.ExcelWriter(PROCESSED_DIR / 'cleaned_data.xlsx') as writer:
    finance_clean.to_excel(writer, sheet_name='finance_indicators', index=False)
    sources_df.to_excel(writer, sheet_name='financing_sources', index=False)

print("Cleaned data exported to:")
print(f"  - {PROCESSED_DIR / 'finance_indicators_clean.csv'}")
print(f"  - {PROCESSED_DIR / 'financing_sources_clean.csv'}")
print(f"  - {PROCESSED_DIR / 'cleaned_data.xlsx'}")

In [ ]:
print("\nData cleaning complete.")
print("\nNext: Proceed to 03_exploratory_analysis.ipynb")